# Closed-Loop Gain Recovery Test

Synthetic closed-loop demonstration of the `newnucal` calibration pipeline:

1. Build a HERA-like array and a random chromatic sky model
2. Simulate visibilities with `ForwardModel` at several times (Earth rotation) and frequencies
3. Apply known per-frequency gain degeneracies (amplitude, phase, phase gradient)
4. Recover gains with the sky held fixed — verifying the solution reproduces the data
5. Show joint sky + gain recovery starting from a perturbed sky

The key physics being tested: the DPSS spectral constraints on the sky/beam model, combined with Earth rotation that decorrelates sky pixels from beam pixels across time, provide enough information to reduce the residuals.  Note that the recovered gains need not match the injected ones exactly — there are residual degeneracies — but the calibrated model visibilities should match the data.

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from astropy.time import Time
from astropy.coordinates import EarthLocation
import astropy.units as u
import healpy

jax.config.update("jax_enable_x64", False)  # float32 throughout

from newnucal import HERAArray, BeamModel, BeamBasis, SkyBasis, SkyModel, ForwardModel, Calibrator, apply_gains, init_gain_params
from newnucal.basis import BeamBasis, SkyBasis
from newnucal.grid_fitter import GridFitter
from newnucal.simulate import compute_rotation_matrices

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110})

## 1. Array, frequency, and time setup

In [ ]:

# --- Array ---
array = HERAArray.from_hex(hexnum=5, sep=14.6)
print(f"Antennas: {array.nants},  Baselines: {array.nbls}")

# --- Frequencies ---
#freqs = array.freq_array(50e6, 225e6)  # Hz
#nfreq = freqs.size
nfreq = 64
freqs = np.linspace(50e6, 225e6, nfreq, dtype=np.float32)  # Hz
print(f"Freqs: {nfreq}, ({freqs[0]/1e6}--{freqs[-1]/1e6} MHz)")

# --- Times ---
hera_loc = EarthLocation(lat=-30.7215 * u.deg, lon=21.4283 * u.deg, height=1073.0 * u.m)
t0 = Time("2023-03-21T04:00:00", scale="utc")
nhrs = 2
ntime = nhrs * 4
dt = nhrs * 60  * u.min / (ntime + 1)
times = t0 + np.arange(ntime) * dt
print(f"Times: {ntime},  span: {(times[-1] - times[0]).to(u.min):.1f}")

# --- Rotation matrices ---
rot_matrices = compute_rotation_matrices(times, hera_loc)
print(f"rot_matrices shape: {rot_matrices.shape}")


## 2. Beam and sky model

In [ ]:
sky_nside = 32
beam_nside = 16

# Beam: Airy disk (HERA dish diameter 14.6 m), DPSS eta_max = 20 ns
#beam_basis = BeamBasis.from_dpss(freqs, eta_max=20e-9)
beam_basis = BeamBasis.from_file('beam_basis_airy.npz', new_freqs=freqs)
beam_model = BeamModel(nside=beam_nside, freqs=freqs, basis=beam_basis)
print(f"Beam Nmodes: {beam_model.nmodes}")

# Sky: random power-law HEALPix map
npix_sky = healpy.nside2npix(sky_nside)
rng = np.random.default_rng(42)

# Build a smooth chromatic sky: per-pixel spectral index drawn from N(-0.7, 0.1)
ref_freq = 150e6
spectral_indices = rng.normal(-0.7, 0.1, npix_sky).astype(np.float32)
ref_flux = rng.exponential(scale=1.0, size=npix_sky).astype(np.float32)
# flux_true[npix, nfreq]
flux_true = ref_flux[:, None] * (freqs[None, :] / ref_freq) ** spectral_indices[:, None]

# Project onto sky DPSS basis
#sky_basis = SkyBasis.from_dpss(freqs, eta_max=40e-9)
sky_basis = SkyBasis.from_file('sky_basis_gsm_gleam.npz', new_freqs=freqs)
sky_coeffs_true = jnp.array(sky_basis.project(flux_true), dtype=jnp.float32)
sky_model  = SkyModel(nside=sky_nside, freqs=freqs, basis=sky_basis)
print(f"Sky Nmodes: {sky_model.nmodes}")

## 3. Simulate true visibilities and apply known gain perturbations

In [ ]:
fwd = ForwardModel(array, sky_model, beam_model, freqs, eps=1e-5)

print("Simulating true visibilities (this compiles the JIT on first run)...")
vis_true = fwd.simulate(sky_coeffs_true, jnp.array(rot_matrices))
print(f"vis_true shape: {vis_true.shape},  dtype: {vis_true.dtype}")
print(f"Mean |vis|: {float(jnp.abs(vis_true).mean()):.4f}")

In [ ]:
# --- True gain perturbations (per time and frequency) ---
# Gains vary smoothly in both time and frequency, mimicking slow ionospheric drift.

t_norm = np.linspace(0, 1, ntime)[:, None]   # (ntime, 1)  — normalised time axis
f_norm = np.linspace(0, 1, nfreq)[None, :]   # (1, nfreq) — normalised frequency axis

# log_amp: ~5% amplitude variation across band, with ~2% slow drift in time
true_log_amp = (
    0.05 * np.cos(2 * np.pi * f_norm)
    + 0.02 * np.sin(2 * np.pi * t_norm)
).astype(np.float32)  # (ntime, nfreq)

# phase: smooth variation in frequency, slow drift in time
true_phase = (
    0.15 * np.sin(2 * np.pi * f_norm)
    + 0.05 * np.cos(2 * np.pi * t_norm)
).astype(np.float32)  # (ntime, nfreq)

# phi: small phase gradients with independent time and frequency variation
true_phi = np.zeros((ntime, 2, nfreq), dtype=np.float32)
true_phi[:, 0, :] = (                               # East
    1e-4 * np.cos(2 * np.pi * f_norm)
    + 3e-5 * np.sin(2 * np.pi * t_norm)
)
true_phi[:, 1, :] = (                               # North
    5e-5 * np.sin(2 * np.pi * f_norm)
    + 2e-5 * np.cos(2 * np.pi * t_norm)
)

true_log_amp_j = jnp.array(true_log_amp)
true_phase_j   = jnp.array(true_phase)
true_phi_j     = jnp.array(true_phi)

vis_data = apply_gains(vis_true, true_log_amp_j, true_phase_j, true_phi_j,
                       jnp.array(array.bls, dtype=jnp.float32))

print(f"vis_data shape: {vis_data.shape}")
print(f"log_amp RMS across times: {float(jnp.std(jnp.array(true_log_amp), axis=0).mean()):.4f}")
print(f"phi_E  RMS across times:  {float(jnp.std(jnp.array(true_phi[:, 0, :]), axis=0).mean()):.2e}")

In [ ]:
cal = Calibrator(
    array=array,
    sky_model=sky_model,
    beam_model=beam_model,
    freqs=freqs,
    rot_matrices=rot_matrices,
    data=vis_data,
    eps=1e-5,
)

In [ ]:
#bm_wgts = cal.get_sky_beam_weighting()
#threshold = bm_wgts.max() / 100
#sky_mask = bm_wgts > threshold
#cal.apply_sky_mask(sky_mask)
#beam_mask = cal.build_beam_mask_from_sky_pixels(sky_mask)
beam_mask = cal.build_beam_mask_altitude(60)
sky_mask = cal.build_sky_mask_from_beam_pixels(beam_mask)
cal.apply_sky_mask(sky_mask)
cal.apply_beam_mask(beam_mask)

In [ ]:
fig = plt.figure()
#healpy.mollview(bm_wgts, fig=fig, sub=(2, 1, 1), title="Sky Beam Weighting", cmap="plasma")
#healpy.mollview(sky_mask,fig=fig, sub=(2, 1, 2), title="Sky Pixel Mask", cmap="gray")
healpy.mollview(sky_mask,fig=fig, title="Sky Pixel Mask", cmap="gray")

In [ ]:
fig = plt.figure()
healpy.orthview(beam_mask, rot=(0, 90, 0), half_sky=True, fig=fig, title="Beam Mask", cmap="gray")

## 5. Data reproduction: baseline spectra and loss

In [ ]:
freq_mhz = freqs / 1e6
t_show = 0
bls_j    = jnp.array(array.bls, dtype=jnp.float32)
beam_j   = jnp.array(beam_model.coeffs, dtype=jnp.float32)
bl_indices = [0, array.nbls // 4, array.nbls // 2, array.nbls - 1]

## 6. Joint sky + gain + beam recovery (perturbed start)

Start the sky from a perturbed version of the truth (30% Gaussian noise on DPSS coefficients) and the beam from a perturbed version (10% noise). Jointly optimize sky, gains, and beam using `fit_alternating_dirty` with `solve_every` scheduling. This tests whether Earth rotation + DPSS spectral constraints are sufficient to disentangle all three components.

In [ ]:
rng2 = np.random.default_rng(99)
sky_coeffs_perturbed = sky_coeffs_true + 0.30 * jnp.array(
    rng2.standard_normal(sky_coeffs_true.shape).astype(np.float32)
) * float(jnp.abs(sky_coeffs_true).mean())

# Perturb the beam coefficients by 10% Gaussian noise
beam_coeffs_true = jnp.array(beam_model.coeffs, dtype=jnp.float32)
rng3 = np.random.default_rng(77)
beam_coeffs_perturbed = beam_coeffs_true + 0.30 * jnp.array(
    rng3.standard_normal(beam_coeffs_true.shape).astype(np.float32)
) * float(jnp.abs(beam_coeffs_true).mean())

print(f"Sky perturbation RMS / signal RMS: "
      f"{float(jnp.std(sky_coeffs_perturbed - sky_coeffs_true)) / float(jnp.std(sky_coeffs_true)):.2f}")
print(f"Beam perturbation RMS / signal RMS: "
      f"{float(jnp.std(beam_coeffs_perturbed - beam_coeffs_true)) / float(jnp.std(beam_coeffs_true)):.2f}")

prms0 = {
    "sky_coeffs":  sky_coeffs_perturbed,
    "beam_coeffs": beam_coeffs_perturbed,
    **init_gain_params(ntime, nfreq),
}

In [ ]:
#%load_ext snakeviz
#%%snakeviz
prms, _ = cal.fit_joint_sky_beam_dirty(
    prms0,
    n_iter=60,
    joint_initial_step=[1.0, 8.0, 16.0],
    joint_anderson_history=4,
    joint_aa_start=2,
    joint_aa_damping=0.45,
    joint_aa_ridge=1e-2,
    solve_every={
        'gains':             8,   # effectively disable routine gains
        'gains_max':         8,   # do not force them either
    },
    check_every=3,
    verbose=True,
)

In [ ]:
vis_joint_before = cal.simulate(prms0)
vis_joint_after  = cal.simulate(prms)

loss_joint_before = cal.calc_loss(prms0)
loss_joint_after  = cal.calc_loss(prms)
print(f"Stage 2 loss — before: {loss_joint_before:.4e}   after: {loss_joint_after:.4e}")
print(f"  reduction: {loss_joint_before / loss_joint_after:.1f}x")

fig, axes = plt.subplots(len(bl_indices), 2, figsize=(12, 2.8 * len(bl_indices)), sharex=True)
fig.suptitle(
    f"Stage 2: Baseline spectra (t={t_show})  —  loss {loss_joint_before:.2e} → {loss_joint_after:.2e}",
    fontsize=11,
)
for row, bi in enumerate(bl_indices):
    bl_len = float(jnp.linalg.norm(bls_j[bi, :2]))

    ax = axes[row, 0]
    ax.semilogy(freq_mhz, jnp.abs(vis_data)[t_show, :, bi],         "k-",  lw=1.5, label="Data")
    ax.semilogy(freq_mhz, jnp.abs(vis_data - vis_joint_before)[t_show, :, bi], "b--", lw=1,   label="Before", alpha=0.7)
    ax.semilogy(freq_mhz, jnp.abs(vis_data - vis_joint_after)[t_show, :, bi],  "r--", lw=1.5, label="After")
    ax.set_ylabel(f"bl {bi} ({bl_len:.0f}m)\n|V|")
    if row == 0: ax.legend(fontsize=8)
    if row == len(bl_indices) - 1: ax.set_xlabel("Frequency (MHz)")

    ax = axes[row, 1]
    ax.plot(freq_mhz, jnp.angle(vis_data)[t_show, :, bi],         "k-",  lw=1.5, label="Data")
    ax.plot(freq_mhz, jnp.angle(vis_joint_before)[t_show, :, bi], "b--", lw=1,   label="Before", alpha=0.7)
    ax.plot(freq_mhz, jnp.angle(vis_joint_after)[t_show, :, bi],  "r--", lw=1.5, label="After")
    ax.set_ylabel(f"bl {bi} ({bl_len:.0f}m)\narg(V) (rad)")
    if row == 0: ax.legend(fontsize=8)
    if row == len(bl_indices) - 1: ax.set_xlabel("Frequency (MHz)")

plt.tight_layout(); plt.show()

## 7. Recovered sky map

In [ ]:
# Reconstruct sky flux from DPSS coefficients at a reference frequency
ifreq_ref = np.argmin(np.abs(freqs - ref_freq))

sky_true_map = np.array(flux_true[:, ifreq_ref])
# Deproject: unsolved pixels keep perturbed values; mask zeroes them out
sky_rec_full = np.array(sky_basis.deproject(np.asarray(prms["sky_coeffs"])))[:, ifreq_ref]
sky_rec_map  = cal.pixel_mask * sky_rec_full
sky_diff_map = cal.pixel_mask * (sky_rec_full - sky_true_map)

vmax = np.percentile(sky_true_map[sky_mask], 99)
vmin = 0.0
diff_scale = np.percentile(np.abs(sky_diff_map[sky_mask]), 99)

fig = plt.figure(figsize=(12, 9))
healpy.mollview(sky_true_map, fig=fig, sub=(3, 1, 1),
                title=f"True sky  ({ref_freq/1e6:.0f} MHz)", min=vmin, max=vmax,
                unit="Jy/pix", cmap="inferno")
healpy.mollview(sky_rec_map,  fig=fig, sub=(3, 1, 2),
                title="Recovered sky  (joint fit, solved pixels only)", min=vmin, max=vmax,
                unit="Jy/pix", cmap="inferno")
healpy.mollview(sky_diff_map, fig=fig, sub=(3, 1, 3),
                title="Residual (recovered − true, solved pixels)", min=-diff_scale, max=diff_scale,
                unit="Jy/pix", cmap="RdBu_r")
plt.show()

## 8. Recovered beam map and spectra

In [ ]:
# Reconstruct beam spectra from DPSS coefficients: (npix_beam, nfreq)
beam_spec_true     = np.array(beam_basis.deproject(np.asarray(beam_coeffs_true)))
beam_spec_perturb  = np.array(beam_basis.deproject(np.asarray(beam_coeffs_perturbed)))
beam_spec_rec      = np.array(beam_basis.deproject(np.asarray(prms["beam_coeffs"])))

rms_before = float(np.sqrt(np.mean((beam_spec_perturb - beam_spec_true) ** 2)))
rms_after  = float(np.sqrt(np.mean((beam_spec_rec     - beam_spec_true) ** 2)))
rms_signal = float(np.sqrt(np.mean(beam_spec_true ** 2)))
print(f"Beam RMS error — before: {rms_before:.4e}  after: {rms_after:.4e}  "
      f"(signal: {rms_signal:.4e})")
print(f"  relative: {rms_before/rms_signal:.3f} → {rms_after/rms_signal:.3f}")

# --- HEALPix maps at reference frequency ---
ifreq_ref = np.argmin(np.abs(freqs - ref_freq))

vmax_b = np.percentile(beam_spec_true[:, ifreq_ref], 99)

fig = plt.figure(figsize=(12, 10))
healpy.orthview(beam_spec_true[:, ifreq_ref], rot=(0, 90, 0), half_sky=True,
                fig=fig, sub=(3, 1, 1),
                title=f"True beam  ({ref_freq/1e6:.0f} MHz)",
                min=0, max=vmax_b, cmap="inferno")
healpy.orthview(beam_spec_rec[:, ifreq_ref], rot=(0, 90, 0), half_sky=True,
                fig=fig, sub=(3, 1, 2),
                title="Recovered beam  (joint fit)",
                min=0, max=vmax_b, cmap="inferno")
diff_b = beam_spec_rec[:, ifreq_ref] - beam_spec_true[:, ifreq_ref]
dscale = np.percentile(np.abs(diff_b), 99)
healpy.orthview(diff_b, rot=(0, 90, 0), half_sky=True,
                fig=fig, sub=(3, 1, 3),
                title="Residual (recovered − true)",
                min=-dscale, max=dscale, cmap="RdBu_r")
plt.show()

# --- Spectra at the peak-beam pixel and a few others ---
peak_px = int(np.argmax(beam_spec_true[:, ifreq_ref]))
sample_pxs = [peak_px,
               int(np.argsort(beam_spec_true[:, ifreq_ref])[-beam_spec_true.shape[0]//4]),
               int(np.argsort(beam_spec_true[:, ifreq_ref])[-beam_spec_true.shape[0]//2])]

fig, axes = plt.subplots(1, len(sample_pxs), figsize=(13, 4), sharey=False)
fig.suptitle("Beam spectra at selected pixels: true vs. perturbed vs. recovered", fontsize=11)
for ax, px in zip(axes, sample_pxs):
    ax.plot(freq_mhz, beam_spec_true[px],    "k-",  lw=2,   label="True")
    ax.plot(freq_mhz, beam_spec_perturb[px], "b--", lw=1.2, label="Perturbed", alpha=0.7)
    ax.plot(freq_mhz, beam_spec_rec[px],     "r--", lw=1.5, label="Recovered")
    ax.set_title(f"pixel {px}")
    ax.set_xlabel("Frequency (MHz)")
    ax.set_ylabel("Beam amplitude")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()